In [2]:
from vizdoom import *
import numpy as np
import random
import time

In [ ]:
# Demo with random actions
EPISODES = 5

game = DoomGame()
game.load_config('ViZDoom/scenarios/basic.cfg')
game.init()

actions = np.identity(3, dtype=np.uint8)

print('REWARDS')
print('--------------------')
for episode in range(EPISODES):
    game.new_episode()

    while not game.is_episode_finished():
        state = game.get_state()
        frame = state.screen_buffer
        info = state.game_variables
        reward = game.make_action(random.choice(actions))
        time.sleep(0.02)
    
    print(f'Episode {episode + 1} = {game.get_total_reward()}')
    time.sleep(2)

game.close()

REWARDS
--------------------
Episode 1 = -249.0
Episode 2 = -197.0
Episode 3 = -375.0
Episode 4 = -47.0
Episode 5 = 70.0


In [3]:
from gymnasium import Env
from gymnasium.spaces import Discrete, Box
import cv2

In [4]:
class VizDoomGym(Env):
    def __init__(self, render=False):
        super().__init__()
        self.game = DoomGame()
        self.game.load_config('ViZDoom/scenarios/basic.cfg')
        self.game.set_window_visible(render)
        self.game.init()

        self.observation_space = Box(low=0, high=255, shape=(100, 160, 1), dtype=np.uint8)
        self.action_space = Discrete(3)

    def step(self, action):
        actions = np.identity(3, dtype=np.uint8)
        reward = self.game.make_action(actions[action], 4) # Skip 4 frames
        state = self.game.get_state()
        done = self.game.is_episode_finished()
        
        if state:
            frame = state.screen_buffer
            frame = self.grayscale(frame)
            info = state.game_variables
        else:
            frame = np.zeros(self.observation_space.shape)
            info = 0

        return frame, reward, done, info

    def render(self):
        pass

    def grayscale(self, observation):
        frame = cv2.cvtColor(np.moveaxis(observation, 0, -1), cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (160, 100), interpolation=cv2.INTER_CUBIC)
        frame = np.reshape(frame, (100, 160, 1))
        
        return frame

    def reset(self):
        self.game.new_episode()
        frame = self.game.get_state().screen_buffer

        return self.grayscale(frame)

    def close(self):
        self.game.close()

In [5]:
env = VizDoomGym(render=True)

In [19]:
env.reset()

array([[[80],
        [87],
        [85],
        ...,
        [75],
        [80],
        [62]],

       [[54],
        [54],
        [52],
        ...,
        [57],
        [48],
        [60]],

       [[75],
        [50],
        [38],
        ...,
        [59],
        [54],
        [36]],

       ...,

       [[92],
        [92],
        [94],
        ...,
        [85],
        [83],
        [89]],

       [[97],
        [97],
        [97],
        ...,
        [92],
        [92],
        [92]],

       [[95],
        [90],
        [92],
        ...,
        [91],
        [93],
        [86]]], shape=(100, 160, 1), dtype=uint8)

In [28]:
env.step(2)

(array([[[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],
 
        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],
 
        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],
 
        ...,
 
        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],
 
        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],
 
        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]]], shape=(100, 160, 1)),
 99.0,
 True,
 0)

In [29]:
env.close()